# 01 — Cleaning & Dataset Final Laporan Kunjungan Posyandu Kota Banjarmasin (2023–2026)

**Mata kuliah:** Visualisasi Data · **Nama:** Muhammad Dzul Fathi Ahyan · **NIM:** 2410817210011

Notebook ini mengubah 4 sheet tahunan (2023, 2024, 2025, 2026) yang berbentuk **tabel lebar** (305 kolom)
menjadi **satu dataset format panjang** yang bersih dan siap dipakai di Tableau, lalu menghitung tabel KPI.

**Aturan cleaning yang disepakati**
1. Periode data: **Januari 2023 – Agustus 2026** (September 2026 baru terisi sebagian → tidak dipakai).
2. Posyandu diidentifikasi dengan **Kecamatan + Kelurahan + Nama Posyandu** (ada nama posyandu yang sama).
3. Nilai **Dirujuk negatif** dianggap kosong (NaN).
4. **% Dilayani dihitung ulang** = Cakupan ÷ Sasaran × 100. Nilai > 100% **ditandai**, tidak dibuang.
5. Kolom **Dirujuk tahun 2023 tidak dipakai** (kosong sepanjang tahun).

**Periode kebijakan**
| Periode | Tahun | Aturan pelayanan |
|---|---|---|
| I | 2023–2024 | Hanya melayani kategori sasaran tertentu |
| II | 2025 | Boleh melayani sasaran di luar kategori |
| III | 2026 | Wajib melayani semua kategori |

**Struktur folder yang dibutuhkan**
```
VISDAT_Posyandu/
├── data/
│   ├── raw/
│   │   └── Laporan Kunjungan Posyandu.xlsx   ← taruh file Excel di sini
│   └── processed/                            ← dibuat otomatis
├── notebooks/
│   └── 01_cleaning_posyandu.ipynb            ← notebook ini
└── outputs/
    └── tables/                               ← dibuat otomatis
```

## 0. Setup

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import openpyxl

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Notebook boleh dijalankan dari folder notebooks/ maupun dari folder utama project
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW_PATH      = ROOT / "data" / "raw" / "Laporan Kunjungan Posyandu.xlsx"
PROCESSED_DIR = ROOT / "data" / "processed"
TABLE_DIR     = ROOT / "outputs" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.exists(), f"File tidak ditemukan: {RAW_PATH}\nTaruh file Excel di folder data/raw/"
print("Root project :", ROOT)
print("File mentah  :", RAW_PATH.name)

Root project : d:\TUGAS\semester 5\Visdat\UTS dan Uas\UAS_VISDAT
File mentah  : Laporan Kunjungan Posyandu.xlsx


In [2]:
# Konstanta struktur sheet
SHEETS   = ["2023", "2024", "2025", "2026"]
BULAN    = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
            "Agustus", "September", "Oktober", "November", "Desember"]
KATEGORI = ["Ibu Hamil", "Bayi Balita", "Usia Sekolah dan Remaja", "Usia Produktif", "Lansia"]
INDIKATOR = ["Sasaran", "Cakupan", "% Dilayani", "Dirujuk", "% Dirujuk"]

N_ID   = 5                      # No, Kecamatan, Kelurahan, Puskesmas, Posyandu
BLOK_K = len(INDIKATOR)         # 5 kolom per kategori
BLOK_B = len(KATEGORI) * BLOK_K # 25 kolom per bulan

PERIODE = {2023: "I", 2024: "I", 2025: "II", 2026: "III"}
PERIODE_LABEL = {
    "I":   "Periode I (2023–2024): Sasaran tertentu",
    "II":  "Periode II (2025): Boleh di luar kategori",
    "III": "Periode III (2026): Wajib semua kategori",
}
BATAS_AKHIR = (2026, 8)          # Agustus 2026

## 1. Membaca sheet & mengubah tabel lebar → format panjang

Setiap sheet memiliki 3 baris header (bulan, kategori, indikator). Posisi baris header bisa berbeda
(sheet 2025 punya 1 baris kosong di atas), jadi baris header dicari otomatis dari sel `No.`.
Struktur header juga divalidasi supaya urutan kategori/indikator pasti benar.

In [3]:
def to_num(v):
    # Ubah isi sel ke angka; teks seperti '68.25' atau '68,25' ikut dikonversi.
    if v is None:
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace(",", ".")
    if s in ("", "-"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


def baca_sheet(ws, tahun):
    data = list(ws.iter_rows(values_only=True))
    h = next(i for i, r in enumerate(data) if r[0] is not None and str(r[0]).strip().startswith("No"))
    row_bulan, row_kat, row_ind = data[h], data[h + 1], data[h + 2]

    # Validasi struktur header
    for bi, b in enumerate(BULAN):
        label = str(row_bulan[N_ID + bi * BLOK_B]).strip().replace("Nopember", "November")
        assert label == b, f"{tahun}: header bulan {b} tidak cocok ({label})"
    for ki, k in enumerate(KATEGORI):
        assert str(row_kat[N_ID + ki * BLOK_K]).strip() == k, f"{tahun}: header kategori {k} tidak cocok"
    for j, ind in enumerate(INDIKATOR):
        assert str(row_ind[N_ID + j]).strip() == ind, f"{tahun}: header indikator {ind} tidak cocok"

    records = []
    for r in data[h + 3:]:
        if r[1] is None or r[4] is None:      # baris kosong / baris total
            continue
        ident = {
            "Tahun": tahun,
            "Kecamatan": str(r[1]).strip(),
            "Kelurahan": str(r[2]).strip(),
            "Puskesmas": str(r[3]).strip(),
            "Posyandu":  str(r[4]).strip(),
        }
        for bi, b in enumerate(BULAN):
            for ki, k in enumerate(KATEGORI):
                c = N_ID + bi * BLOK_B + ki * BLOK_K
                vals = [to_num(r[c + j]) if c + j < len(r) else np.nan for j in range(BLOK_K)]
                records.append({**ident, "BulanNo": bi + 1, "Bulan": b, "Kategori": k,
                                **dict(zip(INDIKATOR, vals))})
    return pd.DataFrame(records)


wb = openpyxl.load_workbook(RAW_PATH, read_only=True, data_only=True)
raw = pd.concat([baca_sheet(wb[s], int(s)) for s in SHEETS], ignore_index=True)
wb.close()

print("Ukuran data panjang (semua bulan):", raw.shape)
raw.head()

Ukuran data panjang (semua bulan): (106020, 13)


,Tahun,Kecamatan,Kelurahan,Puskesmas,Posyandu,BulanNo,Bulan,Kategori,Sasaran,Cakupan,% Dilayani,Dirujuk,% Dirujuk
0,2023,BANJARMASIN BARAT,BASIRIH,BASIRIH BARU,AMPERA BASIRIH,1,Januari,Ibu Hamil,5.0,3.0,60.00,NaN,NaN
1,2023,BANJARMASIN BARAT,BASIRIH,BASIRIH BARU,AMPERA BASIRIH,1,Januari,Bayi Balita,63.0,43.0,68.25,NaN,NaN
2,2023,BANJARMASIN BARAT,BASIRIH,BASIRIH BARU,AMPERA BASIRIH,1,Januari,Usia Sekolah dan Remaja,NaN,NaN,NaN,NaN,NaN
3,2023,BANJARMASIN BARAT,BASIRIH,BASIRIH BARU,AMPERA BASIRIH,1,Januari,Usia Produktif,NaN,NaN,NaN,NaN,NaN
4,2023,BANJARMASIN BARAT,BASIRIH,BASIRIH BARU,AMPERA BASIRIH,1,Januari,Lansia,NaN,NaN,NaN,NaN,NaN


In [4]:
# Jumlah baris posyandu per tahun (sebelum cleaning)
(raw.groupby("Tahun").size() // (len(BULAN) * len(KATEGORI))).rename("Baris posyandu").to_frame()

,Baris posyandu
Tahun,
2023,395
2024,395
2025,487
2026,490


## 2. Profil kualitas data (sebelum cleaning)

Dicatat dulu masalah yang ada, supaya bisa ditulis di laporan bagian **Hasil pengolahan data**.

In [5]:
ada_isi = raw[["Sasaran", "Cakupan"]].notna().any(axis=1)
pct_excel_over = raw["% Dilayani"] > 100

profil = pd.DataFrame({
    "Sel terisi (Sasaran/Cakupan)":  raw[ada_isi].groupby("Tahun").size(),
    "Dirujuk terisi":                raw.groupby("Tahun")["Dirujuk"].apply(lambda s: s.notna().sum()),
    "Dirujuk negatif":               raw.groupby("Tahun")["Dirujuk"].apply(lambda s: (s < 0).sum()),
    "% Dilayani (Excel) > 100":      raw[pct_excel_over].groupby("Tahun").size(),
    "Cakupan > 0 tapi Sasaran 0/kosong":
        raw[(raw["Cakupan"] > 0) & ~(raw["Sasaran"] > 0)].groupby("Tahun").size(),
}).fillna(0).astype(int)
profil

,Sel terisi (Sasaran/Cakupan),Dirujuk terisi,Dirujuk negatif,% Dilayani (Excel) > 100,Cakupan > 0 tapi Sasaran 0/kosong
Tahun,,,,,
2023,11618,0,0,36,1
2024,12844,5501,0,41,14
2025,28435,13635,3,76,389
2026,19306,8204,69,49,185


In [6]:
# Cek bulan terakhir yang terisi pada 2026 → dasar memotong data di Agustus 2026
raw[raw["Tahun"] == 2026].groupby(["BulanNo", "Bulan"])["Cakupan"].sum().to_frame("Total Cakupan 2026")

,,Total Cakupan 2026
BulanNo,Bulan,
1,Januari,37007.0
2,Februari,37671.0
3,Maret,36575.0
4,April,38432.0
5,Mei,38065.0
6,Juni,39521.0
7,Juli,38659.0
8,Agustus,39998.0
9,September,14482.0


## 3. Cleaning

In [7]:
df = raw.copy()

# (1) Batasi periode: Januari 2023 – Agustus 2026
th, bl = BATAS_AKHIR
df = df[(df["Tahun"] < th) | ((df["Tahun"] == th) & (df["BulanNo"] <= bl))].copy()

# Rapikan teks wilayah (Title Case agar enak dibaca di Tableau)
for col in ["Kecamatan", "Kelurahan", "Puskesmas", "Posyandu"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip().str.title()

# (2) ID posyandu = Kecamatan + Kelurahan + Nama
df["Posyandu_ID"] = (df["Kecamatan"] + " | " + df["Kelurahan"] + " | " + df["Posyandu"]).str.upper()
dup = df.duplicated(["Posyandu_ID", "Tahun", "BulanNo", "Kategori"]).sum()
print("Duplikat kunci (ID, tahun, bulan, kategori):", dup)

# (3) Dirujuk negatif -> NaN
n_neg = (df["Dirujuk"] < 0).sum()
df.loc[df["Dirujuk"] < 0, "Dirujuk"] = np.nan

# (5) Dirujuk 2023 tidak dipakai
df.loc[df["Tahun"] == 2023, "Dirujuk"] = np.nan

# Sasaran/Cakupan negatif (jika ada) juga dianggap kosong
for col in ["Sasaran", "Cakupan"]:
    df.loc[df[col] < 0, col] = np.nan

# (4) Hitung ulang persentase
df["Pct_Dilayani"] = np.where(df["Sasaran"] > 0, df["Cakupan"] / df["Sasaran"] * 100, np.nan)
df["Pct_Dirujuk"]  = np.where((df["Cakupan"] > 0) & df["Dirujuk"].notna(),
                              df["Dirujuk"] / df["Cakupan"] * 100, np.nan)

# Flag kualitas data (ditandai, tidak dibuang)
df["Flag_Over100"]       = df["Pct_Dilayani"] > 100
df["Flag_Sasaran_Nol"]   = (df["Cakupan"] > 0) & ~(df["Sasaran"] > 0)
df["Flag_Tidak_Melapor"] = df["Sasaran"].isna() & df["Cakupan"].isna()
df["Aktif_Melayani"]     = df["Cakupan"] > 0

# Variabel turunan
df["Periode"]       = df["Tahun"].map(PERIODE)
df["Periode_Label"] = df["Periode"].map(PERIODE_LABEL)
df["Tanggal"]       = pd.to_datetime(dict(year=df["Tahun"], month=df["BulanNo"], day=1))

print("Dirujuk negatif diubah ke NaN:", n_neg)
print("Ukuran data setelah dipotong Jan 2023–Agu 2026:", df.shape)

Duplikat kunci (ID, tahun, bulan, kategori): 0
Dirujuk negatif diubah ke NaN: 72
Ukuran data setelah dipotong Jan 2023–Agu 2026: (96220, 23)


In [8]:
# Susun kolom akhir
kolom_final = ["Tanggal", "Tahun", "BulanNo", "Bulan", "Periode", "Periode_Label",
               "Kecamatan", "Kelurahan", "Puskesmas", "Posyandu", "Posyandu_ID", "Kategori",
               "Sasaran", "Cakupan", "Pct_Dilayani", "Dirujuk", "Pct_Dirujuk",
               "Aktif_Melayani", "Flag_Over100", "Flag_Sasaran_Nol", "Flag_Tidak_Melapor"]
clean = (df[kolom_final]
         .sort_values(["Tahun", "BulanNo", "Kategori", "Kecamatan", "Kelurahan", "Posyandu"])
         .reset_index(drop=True))
clean["Pct_Dilayani"] = clean["Pct_Dilayani"].round(2)
clean["Pct_Dirujuk"]  = clean["Pct_Dirujuk"].round(2)
clean.head()

,Tanggal,Tahun,BulanNo,Bulan,Periode,Periode_Label,Kecamatan,Kelurahan,Puskesmas,Posyandu,Posyandu_ID,Kategori,Sasaran,Cakupan,Pct_Dilayani,Dirujuk,Pct_Dirujuk,Aktif_Melayani,Flag_Over100,Flag_Sasaran_Nol,Flag_Tidak_Melapor
0,2023-01-01,2023,1,Januari,I,Periode I (2023–2024): Sasaran tertentu,Banjarmasin Barat,Basirih,Basirih Baru,Ampera Basirih,BANJARMASIN BARAT | BASIRIH | AMPERA BASIRIH,Bayi Balita,63.0,43.0,68.25,NaN,NaN,True,False,False,False
1,2023-01-01,2023,1,Januari,I,Periode I (2023–2024): Sasaran tertentu,Banjarmasin Barat,Basirih,Basirih Baru,Anggrek Basirih,BANJARMASIN BARAT | BASIRIH | ANGGREK BASIRIH,Bayi Balita,55.0,41.0,74.55,NaN,NaN,True,False,False,False
2,2023-01-01,2023,1,Januari,I,Periode I (2023–2024): Sasaran tertentu,Banjarmasin Barat,Basirih,Basirih Baru,Bunga Desa Basirih,BANJARMASIN BARAT | BASIRIH | BUNGA DESA BASIRIH,Bayi Balita,65.0,48.0,73.85,NaN,NaN,True,False,False,False
3,2023-01-01,2023,1,Januari,I,Periode I (2023–2024): Sasaran tertentu,Banjarmasin Barat,Basirih,Basirih Baru,Cempaka Sari Basirih,BANJARMASIN BARAT | BASIRIH | CEMPAKA SARI BAS...,Bayi Balita,69.0,50.0,72.46,NaN,NaN,True,False,False,False
4,2023-01-01,2023,1,Januari,I,Periode I (2023–2024): Sasaran tertentu,Banjarmasin Barat,Basirih,Basirih Baru,Gatra Kencana Basirih,BANJARMASIN BARAT | BASIRIH | GATRA KENCANA BA...,Bayi Balita,145.0,93.0,64.14,NaN,NaN,True,False,False,False


## 4. Ringkasan hasil cleaning (untuk laporan)

In [9]:
ringkas = pd.DataFrame({
    "Posyandu (unik)":       clean.groupby("Tahun")["Posyandu_ID"].nunique(),
    "Kecamatan":             clean.groupby("Tahun")["Kecamatan"].nunique(),
    "Kelurahan":             clean.groupby("Tahun")["Kelurahan"].nunique(),
    "Puskesmas":             clean.groupby("Tahun")["Puskesmas"].nunique(),
    "Bulan":                 clean.groupby("Tahun")["BulanNo"].nunique(),
    "Observasi":             clean.groupby("Tahun").size(),
    "Flag % Dilayani > 100": clean.groupby("Tahun")["Flag_Over100"].sum(),
    "Flag Sasaran 0":        clean.groupby("Tahun")["Flag_Sasaran_Nol"].sum(),
})
print("Total observasi:", len(clean))
ringkas

Total observasi: 96220


,Posyandu (unik),Kecamatan,Kelurahan,Puskesmas,Bulan,Observasi,Flag % Dilayani > 100,Flag Sasaran 0
Tahun,,,,,,,,
2023,395,5,52,27,12,23700,36,1
2024,395,5,52,28,12,23700,41,14
2025,487,5,52,28,12,29220,77,389
2026,490,5,52,28,8,19600,49,185


## 5. Tabel KPI

Catatan metode: persentase agregat dihitung sebagai **Σ Cakupan ÷ Σ Sasaran × 100** (rata-rata tertimbang),
bukan rata-rata dari persen tiap posyandu, supaya posyandu kecil tidak mendominasi.
Semua perbandingan antartahun memakai bulan yang sama (**Januari–Agustus**) agar adil.

In [10]:
JAN_AGU = clean[clean["BulanNo"] <= 8]

def agregat(data, by):
    g = data.groupby(by, observed=True).agg(
        Sasaran=("Sasaran", "sum"),
        Cakupan=("Cakupan", "sum"),
        Dirujuk=("Dirujuk", "sum"),
        Posyandu_Aktif=("Posyandu_ID", lambda s: s[data.loc[s.index, "Aktif_Melayani"]].nunique()),
    ).reset_index()
    g["Pct_Dilayani"] = (g["Cakupan"] / g["Sasaran"].replace(0, np.nan) * 100).round(2)
    g["Gap_Belum_Terlayani"] = (g["Sasaran"] - g["Cakupan"]).clip(lower=0)
    return g

# 5.1 KPI per tahun (Jan–Agu)
kpi_tahun = agregat(JAN_AGU, ["Tahun", "Periode"])
kpi_tahun["Perubahan_Cakupan_vs_2023_%"] = (
    (kpi_tahun["Cakupan"] / kpi_tahun.loc[kpi_tahun["Tahun"] == 2023, "Cakupan"].iloc[0] - 1) * 100).round(1)
kpi_tahun

,Tahun,Periode,Sasaran,Cakupan,Dirujuk,Posyandu_Aktif,Pct_Dilayani,Gap_Belum_Terlayani,Perubahan_Cakupan_vs_2023_%
0,2023,I,303113.0,171623.0,0.0,395,56.62,131490.0,0.0
1,2024,I,701861.0,180611.0,415.0,395,25.73,521250.0,5.2
2,2025,II,1347729.0,278505.0,222.0,487,20.66,1069224.0,62.3
3,2026,III,2264481.0,305931.0,174.0,482,13.51,1958550.0,78.3


In [11]:
# 5.2 KPI per tahun × kategori (Jan–Agu) + kontribusi terhadap total kunjungan
kpi_kat = agregat(JAN_AGU, ["Tahun", "Periode", "Kategori"])
kpi_kat["Kontribusi_%"] = (kpi_kat["Cakupan"] /
                           kpi_kat.groupby("Tahun")["Cakupan"].transform("sum") * 100).round(1)
base = kpi_kat[kpi_kat["Tahun"] == 2023].set_index("Kategori")["Cakupan"]
kpi_kat["Perubahan_Cakupan_vs_2023_%"] = ((kpi_kat["Cakupan"] / kpi_kat["Kategori"].map(base) - 1) * 100).round(1)
kpi_kat.pivot(index="Kategori", columns="Tahun", values=["Kontribusi_%", "Pct_Dilayani", "Posyandu_Aktif"])

Kontribusi_%                   Pct_Dilayani                      Posyandu_Aktif                     
Tahun                           2023  2024  2025  2026         2023   2024   2025   2026           2023   2024   2025   2026
Kategori                                                                                                                    
Bayi Balita                     82.8  80.2  40.9  35.0        68.07  74.22  60.54  52.12          395.0  395.0  487.0  482.0
Ibu Hamil                        5.4   4.6   2.6   2.0        72.40  72.46  58.45  51.55          366.0  394.0  474.0  459.0
Lansia                           5.1   7.7  14.9  18.2        30.32  22.00  22.65  17.31           48.0   71.0  487.0  482.0
Usia Produktif                   3.7   4.0  34.5  36.2        19.03   1.78  14.31   8.69           45.0   78.0  487.0  482.0
Usia Sekolah dan Remaja          3.0   3.5   7.1   8.6        26.30  29.74   6.76   5.81           29.0   52.0  461.0  463.0

In [12]:
# 5.3 KPI per periode × kategori (Jan–Agu, rata-rata per tahun supaya Periode I yang 2 tahun sebanding)
kpi_periode = agregat(JAN_AGU, ["Periode", "Kategori"])
n_tahun = JAN_AGU.groupby("Periode")["Tahun"].nunique()
for col in ["Sasaran", "Cakupan", "Dirujuk", "Gap_Belum_Terlayani"]:
    kpi_periode[col + "_per_Tahun"] = (kpi_periode[col] / kpi_periode["Periode"].map(n_tahun)).round(0)
kpi_periode["Kontribusi_%"] = (kpi_periode["Cakupan"] /
                               kpi_periode.groupby("Periode")["Cakupan"].transform("sum") * 100).round(1)
kpi_periode

,Periode,Kategori,Sasaran,Cakupan,Dirujuk,Posyandu_Aktif,Pct_Dilayani,Gap_Belum_Terlayani,Sasaran_per_Tahun,Cakupan_per_Tahun,Dirujuk_per_Tahun,Gap_Belum_Terlayani_per_Tahun,Kontribusi_%
0,I,Bayi Balita,403926.0,286945.0,236.0,408,71.04,116981.0,201963.0,143472.0,118.0,58490.0,81.5
1,I,Ibu Hamil,24253.0,17567.0,139.0,408,72.43,6686.0,12126.0,8784.0,70.0,3343.0,5.0
2,I,Lansia,91856.0,22608.0,32.0,79,24.61,69248.0,45928.0,11304.0,16.0,34624.0,6.4
3,I,Usia Produktif,443855.0,13571.0,8.0,98,3.06,430284.0,221928.0,6786.0,4.0,215142.0,3.9
4,I,Usia Sekolah dan Remaja,41084.0,11543.0,0.0,59,28.10,29541.0,20542.0,5772.0,0.0,14770.0,3.3
5,II,Bayi Balita,188275.0,113984.0,44.0,487,60.54,74291.0,188275.0,113984.0,44.0,74291.0,40.9
6,II,Ibu Hamil,12217.0,7141.0,27.0,474,58.45,5076.0,12217.0,7141.0,27.0,5076.0,2.6
7,II,Lansia,183337.0,41528.0,67.0,487,22.65,141809.0,183337.0,41528.0,67.0,141809.0,14.9
8,II,Usia Produktif,671292.0,96082.0,27.0,487,14.31,575210.0,671292.0,96082.0,27.0,575210.0,34.5
9,II,Usia Sekolah dan Remaja,292608.0,19770.0,57.0,461,6.76,272838.0,292608.0,19770.0,57.0,272838.0,7.1


In [13]:
# 5.4 KPI per kecamatan × tahun × kategori (Jan–Agu)
kpi_kec = agregat(JAN_AGU, ["Tahun", "Periode", "Kecamatan", "Kategori"])
kpi_kec[kpi_kec["Tahun"] == 2026].pivot(index="Kecamatan", columns="Kategori", values="Pct_Dilayani")

Kategori,Bayi Balita,Ibu Hamil,Lansia,Usia Produktif,Usia Sekolah dan Remaja
Kecamatan,,,,,
Banjarmasin Barat,48.53,44.45,26.64,12.56,4.59
Banjarmasin Selatan,55.74,62.29,8.68,6.55,4.55
Banjarmasin Tengah,52.10,48.75,21.07,7.99,7.11
Banjarmasin Timur,51.89,57.80,19.33,10.23,6.87
Banjarmasin Utara,51.26,48.35,25.27,9.40,7.13


In [14]:
# 5.5 Tren bulanan per kategori (Jan 2023–Agu 2026) → untuk grafik garis
tren = agregat(clean, ["Tanggal", "Tahun", "BulanNo", "Bulan", "Periode", "Kategori"])
tren.tail(10)

,Tanggal,Tahun,BulanNo,Bulan,Periode,Kategori,Sasaran,Cakupan,Dirujuk,Posyandu_Aktif,Pct_Dilayani,Gap_Belum_Terlayani
210,2026-07-01,2026,7,Juli,III,Bayi Balita,26688.0,13874.0,46.0,463,51.99,12814.0
211,2026-07-01,2026,7,Juli,III,Ibu Hamil,1566.0,832.0,0.0,366,53.13,734.0
212,2026-07-01,2026,7,Juli,III,Lansia,41142.0,7127.0,3.0,457,17.32,34015.0
213,2026-07-01,2026,7,Juli,III,Usia Produktif,165341.0,13534.0,10.0,463,8.19,151807.0
214,2026-07-01,2026,7,Juli,III,Usia Sekolah dan Remaja,59726.0,3292.0,0.0,400,5.51,56434.0
215,2026-08-01,2026,8,Agustus,III,Bayi Balita,23921.0,13866.0,3.0,431,57.97,10055.0
216,2026-08-01,2026,8,Agustus,III,Ibu Hamil,1659.0,768.0,0.0,355,46.29,891.0
217,2026-08-01,2026,8,Agustus,III,Lansia,37720.0,6749.0,5.0,409,17.89,30971.0
218,2026-08-01,2026,8,Agustus,III,Usia Produktif,148532.0,15138.0,6.0,430,10.19,133394.0
219,2026-08-01,2026,8,Agustus,III,Usia Sekolah dan Remaja,51911.0,3477.0,0.0,367,6.70,48434.0


In [15]:
# 5.6 Data per posyandu × tahun × kategori (Jan–Agu) → untuk boxplot sebaran % dilayani
per_posyandu = agregat(JAN_AGU, ["Tahun", "Periode", "Kecamatan", "Kelurahan", "Puskesmas",
                                 "Posyandu", "Posyandu_ID", "Kategori"])
per_posyandu = per_posyandu.drop(columns="Posyandu_Aktif")
per_posyandu["Flag_Over100"] = per_posyandu["Pct_Dilayani"] > 100
per_posyandu.groupby(["Tahun", "Kategori"])["Pct_Dilayani"].median().unstack().round(1)

Kategori,Bayi Balita,Ibu Hamil,Lansia,Usia Produktif,Usia Sekolah dan Remaja
Tahun,,,,,
2023,71.0,81.2,53.5,71.3,52.6
2024,74.2,79.1,65.7,85.4,74.2
2025,63.9,66.7,35.8,38.6,10.9
2026,54.8,55.6,22.6,11.0,5.5


In [16]:
# 5.7 Beban kategori baru vs capaian balita per posyandu (untuk scatter Insight 3)
INTI = ["Bayi Balita", "Ibu Hamil"]
rows = []
for th in [2025, 2026]:
    y = JAN_AGU[JAN_AGU["Tahun"] == th]
    tot  = y.groupby("Posyandu_ID")["Cakupan"].sum()
    baru = y[~y["Kategori"].isin(INTI)].groupby("Posyandu_ID")["Cakupan"].sum()
    bb   = y[y["Kategori"] == "Bayi Balita"].groupby("Posyandu_ID")[["Sasaran", "Cakupan"]].sum()
    info = y.groupby("Posyandu_ID")[["Kecamatan", "Kelurahan", "Posyandu"]].first()
    t = info.assign(Tahun=th, Periode=PERIODE[th],
                    Total_Kunjungan=tot,
                    Porsi_Kategori_Baru=(baru / tot * 100).round(2),
                    Pct_Dilayani_Balita=(bb["Cakupan"] / bb["Sasaran"].replace(0, np.nan) * 100).round(2))
    t = t.dropna(subset=["Porsi_Kategori_Baru", "Pct_Dilayani_Balita"])
    t["Kuartil_Beban"] = pd.qcut(t["Porsi_Kategori_Baru"], 4, labels=["Q1 Rendah", "Q2", "Q3", "Q4 Tinggi"])
    rows.append(t.reset_index())
beban = pd.concat(rows, ignore_index=True)

print(beban.groupby("Tahun")[["Porsi_Kategori_Baru", "Pct_Dilayani_Balita"]].corr().round(3))
beban.groupby(["Tahun", "Kuartil_Beban"], observed=True)["Pct_Dilayani_Balita"].median().unstack().round(1)

                           Porsi_Kategori_Baru  Pct_Dilayani_Balita
Tahun                                                              
2025  Porsi_Kategori_Baru                1.000               -0.402
      Pct_Dilayani_Balita               -0.402                1.000
2026  Porsi_Kategori_Baru                1.000               -0.289
      Pct_Dilayani_Balita               -0.289                1.000


Kuartil_Beban,Q1 Rendah,Q2,Q3,Q4 Tinggi
Tahun,,,,
2025,73.4,66.1,55.6,56.8
2026,61.1,55.2,52.4,42.1


## 6. Validasi angka kunci (harus sama dengan laporan UTS)

In [17]:
k = kpi_kat.set_index(["Tahun", "Kategori"])
cek = {
    "Kontribusi Bayi Balita 2023 (%)":  k.loc[(2023, "Bayi Balita"), "Kontribusi_%"],
    "Kontribusi Bayi Balita 2026 (%)":  k.loc[(2026, "Bayi Balita"), "Kontribusi_%"],
    "Kontribusi Usia Produktif 2023 (%)": k.loc[(2023, "Usia Produktif"), "Kontribusi_%"],
    "Kontribusi Usia Produktif 2026 (%)": k.loc[(2026, "Usia Produktif"), "Kontribusi_%"],
    "Posyandu aktif Usia Produktif 2023": k.loc[(2023, "Usia Produktif"), "Posyandu_Aktif"],
    "Posyandu aktif Usia Produktif 2026": k.loc[(2026, "Usia Produktif"), "Posyandu_Aktif"],
}
pd.Series(cek, name="Nilai").to_frame()

,Nilai
Kontribusi Bayi Balita 2023 (%),82.8
Kontribusi Bayi Balita 2026 (%),35.0
Kontribusi Usia Produktif 2023 (%),3.7
Kontribusi Usia Produktif 2026 (%),36.2
Posyandu aktif Usia Produktif 2023,45.0
Posyandu aktif Usia Produktif 2026,482.0


## 7. Ekspor file

In [18]:
def simpan(data, path):
    # utf-8-sig agar huruf/simbol terbaca benar saat dibuka di Excel
    data.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"✔ {path.relative_to(ROOT)}  ({len(data):,} baris)")

simpan(clean,        PROCESSED_DIR / "posyandu_long_clean.csv")
simpan(kpi_tahun,    TABLE_DIR / "kpi_tahun.csv")
simpan(kpi_kat,      TABLE_DIR / "kpi_tahun_kategori.csv")
simpan(kpi_periode,  TABLE_DIR / "kpi_periode_kategori.csv")
simpan(kpi_kec,      TABLE_DIR / "kpi_kecamatan.csv")
simpan(tren,         TABLE_DIR / "tren_bulanan.csv")
simpan(per_posyandu, TABLE_DIR / "posyandu_per_tahun.csv")
simpan(profil.reset_index(), TABLE_DIR / "profil_kualitas_data.csv")
simpan(beban,        TABLE_DIR / "beban_vs_balita.csv")

# Satu file Excel berisi semua tabel KPI (praktis untuk laporan / Tableau)
with pd.ExcelWriter(TABLE_DIR / "kpi_posyandu.xlsx") as xw:
    for nama, t in {"kpi_tahun": kpi_tahun, "kpi_tahun_kategori": kpi_kat,
                    "kpi_periode_kategori": kpi_periode, "kpi_kecamatan": kpi_kec,
                    "tren_bulanan": tren, "beban_vs_balita": beban, "profil_kualitas": profil.reset_index()}.items():
        t.to_excel(xw, sheet_name=nama, index=False)
print("✔ outputs/tables/kpi_posyandu.xlsx")

✔ data\processed\posyandu_long_clean.csv  (96,220 baris)
✔ outputs\tables\kpi_tahun.csv  (4 baris)
✔ outputs\tables\kpi_tahun_kategori.csv  (20 baris)
✔ outputs\tables\kpi_periode_kategori.csv  (15 baris)
✔ outputs\tables\kpi_kecamatan.csv  (100 baris)
✔ outputs\tables\tren_bulanan.csv  (220 baris)
✔ outputs\tables\posyandu_per_tahun.csv  (8,835 baris)
✔ outputs\tables\profil_kualitas_data.csv  (4 baris)
✔ outputs\tables\beban_vs_balita.csv  (969 baris)
✔ outputs/tables/kpi_posyandu.xlsx


## Kamus data — `posyandu_long_clean.csv`

| Kolom | Keterangan |
|---|---|
| Tanggal | Tanggal awal bulan (untuk sumbu waktu di Tableau) |
| Tahun, BulanNo, Bulan | Waktu pelaporan |
| Periode, Periode_Label | Periode kebijakan I / II / III |
| Kecamatan, Kelurahan, Puskesmas, Posyandu | Identitas wilayah & unit pelayanan |
| Posyandu_ID | Kunci unik posyandu (Kecamatan \| Kelurahan \| Nama) |
| Kategori | Ibu Hamil, Bayi Balita, Usia Sekolah dan Remaja, Usia Produktif, Lansia |
| Sasaran, Cakupan, Dirujuk | Jumlah orang |
| Pct_Dilayani | Cakupan ÷ Sasaran × 100 (dihitung ulang) |
| Pct_Dirujuk | Dirujuk ÷ Cakupan × 100 (dihitung ulang) |
| Aktif_Melayani | True jika Cakupan > 0 |
| Flag_Over100 | True jika % Dilayani > 100 (perlu dicek, tidak dibuang) |
| Flag_Sasaran_Nol | True jika ada Cakupan tetapi Sasaran 0/kosong |
| Flag_Tidak_Melapor | True jika Sasaran dan Cakupan sama-sama kosong |

**Tips Tableau:** untuk % dilayani agregat, buat calculated field `SUM([Cakupan]) / SUM([Sasaran])`,
jangan `AVG([Pct_Dilayani])`.